# end-grad-default-ones-like — ex1: resolve end_grad — default ones_like, else use .array

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `end-grad-default-ones-like`. Running the final beacon cell reports progress against the `Backprop: end-grad ones_like default` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: end-grad ones_like default` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`end-grad-default-ones-like`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "end-grad-default-ones-like"
DD_SUBTOPIC = "Backprop: end-grad ones_like default"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## end-grad ones_like default — quick refresher

PyTorch lets you call `.backward()` on a scalar with no argument:

```
loss = ...
loss.backward()        # same as loss.backward(torch.tensor(1.0))
```

Inside `backprop(end_node, end_grad=None)` the convention is: if `end_grad` is `None`, default to `ones_like(end_node.array)` — a tensor of ones with the same shape as the end node's array.

```python
def backprop(end_node, end_grad=None):
    end_grad_arr = (
        torch.ones_like(end_node.array)
        if end_grad is None
        else end_grad.array
    )
    ...
```

Why ones, not zeros? Because `dL/dL = 1` — when `end_node` IS the loss, its gradient w.r.t. itself is the identity, and 1 is the multiplicative identity that lets every downstream chain-rule product reduce to the actual partial.

Why `ones_like` (matching shape), not `ones(1)`? Because the end node may NOT be a scalar — for a `(B,)` per-sample loss vector you want `dL/dL == eye(B)` collapsed to a `(B,)` of ones.

### Exercise 1 — resolve end_grad — default ones_like, else use .array

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the .backward() entry-point convention: when no end_grad is supplied, default to torch.ones_like(end_node.array); otherwise use end_grad.array as-is.
> Keywords: end-grad, ones-like, backward, default, shape-matching
> ```

**KCs targeted:** `end-grad-default-ones-like`, `buffer-copy-inplace`

Implement `resolve_end_grad(end_node, end_grad)`. This is the FIRST thing `backprop(end_node, end_grad=None)` does — figure out the seed gradient that starts the reverse pass.

Rules:

1. **`end_grad is None` → return `torch.ones_like(end_node.array)`.** Same shape, same dtype as `end_node.array`, all ones. This handles the common `loss.backward()` call where the user means `dL/dL = 1`.

2. **`end_grad` is a `MiniTensor` → return `end_grad.array`.** Unbox; the rest of `backprop` works with raw arrays internally. The shape and dtype must match `end_node.array` — if they don't, raise `AssertionError` with a helpful message:

```python
assert end_grad.array.shape == end_node.array.shape, (
    f'end_grad shape {tuple(end_grad.array.shape)} mismatches '
    f'end_node shape {tuple(end_node.array.shape)}'
)
```

**Why `ones_like` and not `ones(1)`.** The end node may be a scalar (`loss.shape == ()`), a per-sample loss vector (`(B,)`), or a structured object. `ones_like` correctly handles all of them — for a `(B,)` per-sample loss, the seed is `ones(B)`, encoding `dL/dL_i = 1` for every sample independently.

**Why ones, not zeros.** `dL/dL = 1` (the identity). All downstream back fns multiply by this seed; 1 is the multiplicative identity that lets every chain-rule product reduce to the actual partial.

Inputs:
- `end_node`: a `MiniTensor` whose `.array` is a `torch.Tensor`.
- `end_grad`: `None` OR a `MiniTensor`.

Output: a raw `torch.Tensor` (NOT a `MiniTensor`) matching `end_node.array`'s shape and dtype.

Do NOT call `torch.autograd`.

In [ ]:
def resolve_end_grad(end_node: 'MiniTensor', end_grad: 'MiniTensor | None') -> Tensor:
    """Resolve the seed gradient for backprop.

    None  -> torch.ones_like(end_node.array)
    given -> end_grad.array (must match end_node.array shape)
    """
    raise NotImplementedError()


def _test_ex1():
    # --- scalar end node, end_grad=None → ones_like (a 0-D 1.0) ---
    scalar = MiniTensor(t.tensor(3.5))
    g = resolve_end_grad(scalar, None)
    assert isinstance(g, t.Tensor) and not isinstance(g, MiniTensor), (
        'must return raw torch.Tensor, not MiniTensor'
    )
    assert g.shape == (), f'scalar shape: {g.shape}'
    assert g.dtype == scalar.array.dtype, f'dtype must match: {g.dtype} vs {scalar.array.dtype}'
    assert g.item() == 1.0, f'scalar end_grad must be 1.0, got {g.item()}'

    # --- vector end node, end_grad=None → ones of matching shape ---
    vec = MiniTensor(t.tensor([1.0, 2.0, 3.0, 4.0]))
    g = resolve_end_grad(vec, None)
    assert g.shape == (4,)
    assert t.allclose(g, t.ones(4)), f'vec ones: {g}'

    # --- matrix end node, end_grad=None ---
    mat = MiniTensor(t.zeros(3, 5))
    g = resolve_end_grad(mat, None)
    assert g.shape == (3, 5)
    assert t.allclose(g, t.ones(3, 5))

    # --- explicit end_grad → unbox and return as-is ---
    explicit_grad = MiniTensor(t.tensor([0.5, 0.5, 0.5, 0.5]))
    g = resolve_end_grad(vec, explicit_grad)
    assert isinstance(g, t.Tensor) and not isinstance(g, MiniTensor)
    assert t.allclose(g, t.tensor([0.5, 0.5, 0.5, 0.5])), f'explicit grad: {g}'
    # IDENTITY: the returned tensor should be `explicit_grad.array` (not a copy).
    assert g is explicit_grad.array, 'unbox should be identity, not copy'

    # --- shape mismatch raises AssertionError ---
    wrong_shape = MiniTensor(t.zeros(2, 5))  # vec is (4,)
    try:
        resolve_end_grad(vec, wrong_shape)
    except AssertionError as e:
        msg = str(e)
        # Helpful message must mention both shapes.
        assert '(2, 5)' in msg or '2, 5' in msg, f'msg missing end_grad shape: {msg!r}'
        assert '(4,)' in msg or '4,' in msg, f'msg missing end_node shape: {msg!r}'
    else:
        raise AssertionError('shape mismatch should have raised AssertionError')

    # --- dtype preserved by ones_like ---
    fp64_node = MiniTensor(t.tensor([1.0, 2.0], dtype=t.float64))
    g = resolve_end_grad(fp64_node, None)
    assert g.dtype == t.float64, f'ones_like must preserve dtype, got {g.dtype}'

    # --- integer dtype (rare but legal) — still ones_like, dtype preserved ---
    int_node = MiniTensor(t.tensor([1, 2, 3], dtype=t.int32))
    g = resolve_end_grad(int_node, None)
    assert g.dtype == t.int32, f'int32 dtype preserved: {g.dtype}'
    assert t.eq(g, t.ones(3, dtype=t.int32)).all()
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def resolve_end_grad(end_node, end_grad):
    if end_grad is None:
        # default seed: dL/dL = 1, same shape & dtype as end_node.
        return t.ones_like(end_node.array)
    # explicit end_grad — must be a MiniTensor; check shape and unbox.
    assert end_grad.array.shape == end_node.array.shape, (
        f'end_grad shape {tuple(end_grad.array.shape)} mismatches '
        f'end_node shape {tuple(end_node.array.shape)}'
    )
    return end_grad.array
```

**Why `ones_like` is the right default.** The seed of the reverse pass is `d(end_node)/d(end_node)` — the gradient of `end_node` with respect to itself. For a scalar that's `1`. For a non-scalar end node, the user is implicitly asking for the gradient of `end_node.sum()` w.r.t. each input — and the Jacobian of `sum` is `ones_like(input)`. Same answer, easier to implement.

**Why this requires the user to pass `end_grad` when `end_node` is non-scalar in PyTorch.** PyTorch raises if you call `tensor.backward()` on a non-scalar without `gradient=...`. The drill convention is LOOSER — we always default to `ones_like`, which assumes 'sum reduction.' That's fine for educational use but be aware real PyTorch is stricter.

**Why return raw tensor, not MiniTensor.** The rest of `backprop` works with raw `torch.Tensor` accumulators in a `dict[MiniTensor, Tensor]`. Keeping MiniTensors out of the internal accumulator dict avoids double-wrapping and confusion about which fields are populated.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()